In [ ]:
import json
import os

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Circle, Polygon

## Data adapters om json bestanden in te lezen

Eerst de scenario's en stages in laden omzetten naar tabel. De id's verbonden aan een stage zijn bepalend voor welke data later gebruikt wordt uit de andere tabellen

In [ ]:
def input_stages(input_config: dict) -> pd.DataFrame:
    """
    Leest alle scenario JSON bestanden uit scenario folder
    en zet alle stages om naar een platte tabel, met gekoppelde id's.

    Output columns:
    ----------------
    stage_id (PRIMARY KEY)
    stage_label
    scenario_id
    scenario_label
    geometry_id
    decorations_id
    soillayers_id
    waternet_id
    waternet_creator_settings_id
    state_id
    state_correlations_id
    loads_id
    reinforcements_id
    calculationsettings_id
    calculation_id
    content_version
    """

    folder_path = input_config["abs_path"]
    rows = []

    for file_name in os.listdir(folder_path):
        if not file_name.endswith(".json"):
            continue

        file_path = os.path.join(folder_path, file_name)
        with open(file_path, "r") as f:
            data = json.load(f)

        scenario_id = data.get("Id")
        scenario_label = data.get("Label")
        content_version = data.get("ContentVersion")

        # Per stage, koppel alle calculations die in dit scenario staan
        calculations = data.get("Calculations", [])

        for stage in data.get("Stages", []):
            stage_id = stage.get("Id")

            if calculations:
                # Als er meerdere calculations zijn, maak meerdere rijen
                for calc in calculations:
                    rows.append(
                        {
                            "stage_id": stage_id,
                            "stage_label": stage.get("Label"),
                            "scenario_id": scenario_id,
                            "scenario_label": scenario_label,
                            "geometry_id": stage.get("GeometryId"),
                            "decorations_id": stage.get("DecorationsId"),
                            "soillayers_id": stage.get("SoilLayersId"),
                            "waternet_id": stage.get("WaternetId"),
                            "waternet_creator_settings_id": stage.get(
                                "WaternetCreatorSettingsId"
                            ),
                            "state_id": stage.get("StateId"),
                            "state_correlations_id": stage.get("StateCorrelationsId"),
                            "loads_id": stage.get("LoadsId"),
                            "reinforcements_id": stage.get("ReinforcementsId"),
                            "calculationsettings_id": calc.get("CalculationSettingsId"),
                            "calculation_id": calc.get("Id"),
                            "content_version": content_version,
                        }
                    )
            else:
                # Geen calculation → gewoon stage rij met None
                rows.append(
                    {
                        "stage_id": stage_id,
                        "stage_label": stage.get("Label"),
                        "scenario_id": scenario_id,
                        "scenario_label": scenario_label,
                        "geometry_id": stage.get("GeometryId"),
                        "decorations_id": stage.get("DecorationsId"),
                        "soillayers_id": stage.get("SoilLayersId"),
                        "waternet_id": stage.get("WaternetId"),
                        "waternet_creator_settings_id": stage.get(
                            "WaternetCreatorSettingsId"
                        ),
                        "state_id": stage.get("StateId"),
                        "state_correlations_id": stage.get("StateCorrelationsId"),
                        "loads_id": stage.get("LoadsId"),
                        "reinforcements_id": stage.get("ReinforcementsId"),
                        "calculationsettings_id": None,
                        "calculation_id": None,
                        "content_version": content_version,
                    }
                )

    df = pd.DataFrame(rows)
    return df

In [ ]:
input_config = {"abs_path": "data_sets/stix/scenarios"}
df_stages = input_stages(input_config)

df_stages

Nu geometrie tabel maken hierin staan de de lijnen waarmee de lagen gemaakt worden

In [ ]:
def input_geometry(input_config: dict) -> pd.DataFrame:
    """
    Leest alle geometry JSON bestanden in een map
    en zet layers om naar een platte tabel.

    Output columns:
    ----------------
    geometry_id (PRIMARY KEY)
    layer_id
    layer_label
    points  # lijst van XZ coördinaten
    content_version
    """
    folder_path = input_config["abs_path"]
    rows = []

    for file_name in os.listdir(folder_path):
        if not file_name.endswith(".json"):
            continue

        file_path = os.path.join(folder_path, file_name)

        with open(file_path, "r") as f:
            data = json.load(f)

        geometry_id = data.get("Id")
        content_version = data.get("ContentVersion")

        for layer in data.get("Layers", []):
            rows.append(
                {
                    "geometry_id": geometry_id,
                    "layer_id": layer.get("Id"),
                    "layer_label": layer.get("Label"),
                    "points": layer.get("Points"),
                    "content_version": content_version,
                }
            )

    df = pd.DataFrame(rows)
    return df

In [ ]:
df_geometry = input_geometry({"abs_path": "data_sets/stix/geometries"})
df_geometry

In de soillayers worden de lagen gekoppeld aan een soil id

In [ ]:
def input_soillayers(input_config: dict) -> pd.DataFrame:
    """
    Leest alle soillayers JSON bestanden in een map
    en zet deze om naar een flat table.

    Output columns:
    ----------------
    soillayers_id
    layer_id
    soil_id
    content_version
    """

    folder_path = input_config["abs_path"]
    rows = []

    for file_name in os.listdir(folder_path):
        if not file_name.endswith(".json"):
            continue

        file_path = os.path.join(folder_path, file_name)

        with open(file_path, "r") as f:
            data = json.load(f)

        soillayers_id = data.get("Id")
        content_version = data.get("ContentVersion")
        soil_layers = data.get("SoilLayers", [])  # <-- correcte key

        for layer in soil_layers:
            layer_id = layer.get("LayerId")
            soil_id = layer.get("SoilId")
            rows.append(
                {
                    "soillayers_id": soillayers_id,
                    "layer_id": layer_id,
                    "soil_id": soil_id,
                    "content_version": content_version,
                }
            )

    df = pd.DataFrame(rows)
    return df

In [ ]:
config = {"abs_path": "data_sets/stix/soillayers"}

df_soillayers = input_soillayers(config)

df_soillayers

In de soils json worden de soil id's gelinkt aan een grondsoort

In [ ]:
def input_soils(file_path: str) -> pd.DataFrame:
    """
    Leest een losse soils JSON en zet deze om naar een flat DataFrame
    met de belangrijkste properties en een kleur voor plotting.
    """

    # Kleurmap per soil code
    SOIL_COLOR_MAP = {
        "ClaS-0": "darkgreen",
        "PeaU-1": "peru",
        "SanB-2": "lightgreen",
        "ClaU-3	": "green",
        "klei-4": "sandybrown",
        "SanN-5": "yellow",
    }

    with open(file_path, "r") as f:
        data = json.load(f)

    soils = data.get("Soils", [])
    content_version = data.get("ContentVersion")

    rows = []
    for soil in soils:
        mc_adv = soil.get("MohrCoulombAdvancedShearStrengthModel", {})
        su_model = soil.get("SuShearStrengthModel", {})

        code = soil.get("Code")
        color = SOIL_COLOR_MAP.get(code, "gray")  # fallback naar grijs

        rows.append(
            {
                "soil_id": soil.get("Id"),
                "name": soil.get("Name"),
                "code": code,
                "color": color,  # nieuwe kolom!
                "volumetric_weight_above_phreatic_level": soil.get(
                    "VolumetricWeightAbovePhreaticLevel"
                ),
                "volumetric_weight_below_phreatic_level": soil.get(
                    "VolumetricWeightBelowPhreaticLevel"
                ),
                "shear_strength_model_type_above_phreatic_level": soil.get(
                    "ShearStrengthModelTypeAbovePhreaticLevel"
                ),
                "shear_strength_model_type_below_phreatic_level": soil.get(
                    "ShearStrengthModelTypeBelowPhreaticLevel"
                ),
                "mohr_coulomb_advanced_cohesion": mc_adv.get("Cohesion"),
                "mohr_coulomb_advanced_friction_angle": mc_adv.get("FrictionAngle"),
                "su_shear_strength_ratio": su_model.get("ShearStrengthRatio"),
                "su_strength_increase_exponent": su_model.get(
                    "StrengthIncreaseExponent"
                ),
                "content_version": content_version,
            }
        )

    df = pd.DataFrame(rows)
    return df

In [ ]:
file_path = "data_sets/stix/soils.json"
df_soils = input_soils(file_path)
df_soils

In de water net folder worden de waternet_id's gekoppeld aan zx coordinaten voor de verschillende type lijnen

In [ ]:
def input_waternet(input_config: dict) -> pd.DataFrame:
    """
    Leest alle waternet JSON bestanden in een map
    en zet alle HeadLines en ReferenceLines om naar een flat table.

    Output columns:
    ----------------
    waternet_id
    line_type      # "Head" of "Reference"
    line_id
    line_label
    x
    z
    top_headline_id       # alleen bij ReferenceLines
    bottom_headline_id    # alleen bij ReferenceLines
    content_version
    """

    folder_path = input_config["abs_path"]
    rows = []

    for file_name in os.listdir(folder_path):
        if not file_name.endswith(".json"):
            continue

        file_path = os.path.join(folder_path, file_name)
        with open(file_path, "r") as f:
            data = json.load(f)

        waternet_id = data.get("Id")
        content_version = data.get("ContentVersion")

        # HeadLines
        for head in data.get("HeadLines", []):
            line_id = head.get("Id")
            line_label = head.get("Label")
            points = head.get("Points", [])
            for p in points:
                rows.append(
                    {
                        "waternet_id": waternet_id,
                        "line_type": "Head",
                        "line_id": line_id,
                        "line_label": line_label,
                        "x": p.get("X"),
                        "z": p.get("Z"),
                        "top_headline_id": None,
                        "bottom_headline_id": None,
                        "content_version": content_version,
                    }
                )

        # ReferenceLines
        for ref in data.get("ReferenceLines", []):
            line_id = ref.get("Id")
            line_label = ref.get("Label")
            top_id = ref.get("TopHeadLineId")
            bottom_id = ref.get("BottomHeadLineId")
            points = ref.get("Points", [])
            for p in points:
                rows.append(
                    {
                        "waternet_id": waternet_id,
                        "line_type": "Reference",
                        "line_id": line_id,
                        "line_label": line_label,
                        "x": p.get("X"),
                        "z": p.get("Z"),
                        "top_headline_id": top_id,
                        "bottom_headline_id": bottom_id,
                        "content_version": content_version,
                    }
                )

    df = pd.DataFrame(rows)
    return df

In [ ]:
config = {"abs_path": "data_sets/stix/waternets"}

df_waternets = input_waternet(config)

df_waternets

Hier worden de calculationsetting ids gekoppeld aan de reken methode en glijcirkel

In [ ]:
def input_calculationsettings(input_config: dict) -> pd.DataFrame:
    """
    Leest alle calculationsettings JSON bestanden in een map
    en zet deze om naar een flat table.

    Output columns:
    ----------------
    calculationsettings_id
    analysis_type
    calculation_type
    model_factor_mean
    model_factor_std
    circle_center_x
    circle_center_z
    circle_radius
    content_version
    """

    folder_path = input_config["abs_path"]
    rows = []

    for file_name in os.listdir(folder_path):
        if not file_name.endswith(".json"):
            continue

        file_path = os.path.join(folder_path, file_name)

        with open(file_path, "r") as f:
            data = json.load(f)

        calculationsettings_id = data.get("Id")
        analysis_type = data.get("AnalysisType")
        calculation_type = data.get("CalculationType")
        model_factor_mean = data.get("ModelFactorMean")
        model_factor_std = data.get("ModelFactorStandardDeviation")
        content_version = data.get("ContentVersion")

        # Dynamisch actieve methode ophalen
        method_data = data.get(analysis_type, {})

        circle_center_x = None
        circle_center_z = None
        circle_radius = None

        if isinstance(method_data, dict):
            circle = method_data.get("Circle")
            if isinstance(circle, dict):
                center = circle.get("Center", {})
                circle_center_x = center.get("X")
                circle_center_z = center.get("Z")
                circle_radius = circle.get("Radius")

        rows.append(
            {
                "calculationsettings_id": calculationsettings_id,
                "analysis_type": analysis_type,
                "calculation_type": calculation_type,
                "model_factor_mean": model_factor_mean,
                "model_factor_std": model_factor_std,
                "circle_center_x": circle_center_x,
                "circle_center_z": circle_center_z,
                "circle_radius": circle_radius,
                "content_version": content_version,
            }
        )

    df = pd.DataFrame(rows)
    return df

In [ ]:
config = {"abs_path": "data_sets/stix/calculationsettings"}

df_calculationsettings = input_calculationsettings(config)

df_calculationsettings

# Stage ID's linken aan de data uit de vorige tabellen

In [ ]:
def merge_stage_geometries_soils(df_stages, df_geometry, df_soillayers, df_soils):
    # Merge stages met geometry (op geometry_id)
    df_merged = df_stages.merge(
        df_geometry, how="left", left_on="geometry_id", right_on="geometry_id"
    )

    # Merge met soillayers (op layer_id)
    df_merged = df_merged.merge(
        df_soillayers,
        how="left",
        left_on="layer_id",
        right_on="layer_id",
        suffixes=("", "_soillayers"),
    )

    # Merge met soils (op soil_id)
    df_merged = df_merged.merge(
        df_soils,
        how="left",
        left_on="soil_id",
        right_on="soil_id",
        suffixes=("", "_soil"),
    )

    # Kolommen herordenen (alleen bestaande kolommen)
    columns_order = [
        "stage_id",
        "stage_label",
        "scenario_id",
        "scenario_label",
        "geometry_id",
        "layer_id",
        "layer_label",
        "points",
        "soillayers_id",
        "soil_id",
        "name",
        "code",
        "color",
    ]

    # Filter alleen bestaande kolommen
    columns_order = [col for col in columns_order if col in df_merged.columns]

    df_final = df_merged[columns_order]

    return df_final

In [ ]:
df_merged_soil = merge_stage_geometries_soils(
    df_stages, df_geometry, df_soillayers, df_soils
)

df_merged_soil

In [ ]:
def merge_stage_water(
    df_stages: pd.DataFrame, df_waternet: pd.DataFrame
) -> pd.DataFrame:
    """
    Merge stages met waternetlijnen en voeg kleuren toe per type lijn.

    Parameters
    ----------
    df_stages : pd.DataFrame
        Stage-data
    df_waternet : pd.DataFrame
        Waternet-data (met punten en line_type info)

    Returns
    -------
    pd.DataFrame
        Per stage de kernkolommen + alle waterlijnen + kleur per line_type
    """

    # Kolommen uit stages die we willen behouden
    stage_cols = ["stage_id", "stage_label", "scenario_id", "scenario_label"]

    df_stages_light = df_stages[stage_cols + ["waternet_id"]].drop_duplicates()

    # Merge met waternet
    df_merged = df_stages_light.merge(df_waternet, how="left", on="waternet_id")

    # Voeg kleuren toe per line_type
    line_color_map = {
        "Phreatic line (PL 1)": "#ff1493",  # deep pink
        "Head line 3 (PL 3)": "#ff8c00",  # helder donkerblauw
        "Waternet line phreatic line": "#00bfff",  # helder lichtblauw
        "Waternet line lower aquifer": "#00ff7f",  # fel groen
    }

    df_merged["color"] = (
        df_merged["line_label"].map(line_color_map).fillna("#cccccc")
    )  # fallback grijs

    # Sorteren zodat lijnen netjes doorlopen
    df_merged = df_merged.sort_values(
        by=["stage_id", "line_type", "line_id", "x"]
    ).reset_index(drop=True)

    return df_merged

In [ ]:
df_merged_water = merge_stage_water(df_stages, df_waternets)

df_merged_water

In [ ]:
def merge_calculations(df_stages, df_calculationsettings):
    # Merge stages met calculationsettings (op calculationsettings_id)
    df_merged = df_stages.merge(
        df_calculationsettings,
        how="left",
        left_on="calculationsettings_id",
        right_on="calculationsettings_id",
    )

    # Kolommen herordenen (alleen bestaande kolommen)
    columns_order = [
        "stage_id",
        "stage_label",
        "scenario_id",
        "scenario_label",
        "calculationsettings_id",
        "analysis_type",
        "calculation_type",
        "model_factor_mean",
        "model_factor_std",
        "circle_center_x",
        "circle_center_z",
        "circle_radius",
        "content_version",
    ]

    columns_order = [col for col in columns_order if col in df_merged.columns]

    df_final = df_merged[columns_order]

    return df_final

In [ ]:
df_merged_calculations = merge_calculations(df_stages, df_calculationsettings)

df_merged_calculations

# Alles plotten op basis van stage id

In [ ]:
def plot_stage_v1(
    df_merged_soil: pd.DataFrame, df_merged_water: pd.DataFrame, stage_id
):
    stage_id = str(stage_id)
    df_stage = df_merged_soil[df_merged_soil["stage_id"] == stage_id]
    df_stage_water = df_merged_water[df_merged_water["stage_id"] == stage_id]

    if df_stage.empty:
        print(f"Geen soil data gevonden voor stage {stage_id}")
        return
    if df_stage_water.empty:
        print(f"Geen water data gevonden voor stage {stage_id}")

    fig, ax = plt.subplots(figsize=(15, 15))

    # --- Plot soils ---
    plotted_soils = {}
    df_stage = df_stage.sort_values("soil_id")
    for idx, row in df_stage.iterrows():
        points = row["points"]
        if not points:
            continue
        polygon_coords = [(p["X"], p["Z"]) for p in points]
        if polygon_coords[0] != polygon_coords[-1]:
            polygon_coords.append(polygon_coords[0])
        color = row["color"]
        poly = Polygon(
            polygon_coords, closed=True, facecolor=color, edgecolor="k", alpha=1
        )
        ax.add_patch(poly)
        soil_name = row["name"]
        if soil_name not in plotted_soils:
            plotted_soils[soil_name] = color

    # --- Plot waterlijnen ---
    plotted_lines = {}
    for line_id, df_line in df_stage_water.groupby("line_id"):
        xs = df_line["x"].tolist()
        zs = df_line["z"].tolist()
        line_label = df_line["line_label"].iloc[0]
        color = df_line["color"].iloc[0]
        ax.plot(xs, zs, color=color, linewidth=2)
        if line_label not in plotted_lines:
            plotted_lines[line_label] = color

    # --- Legenda soils ---
    soil_handles = [
        plt.Line2D([0], [0], color=color, lw=10) for color in plotted_soils.values()
    ]
    soil_labels = list(plotted_soils.keys())
    soil_legend = ax.legend(
        soil_handles,
        soil_labels,
        title="Soil type",
        loc="upper right",
        bbox_to_anchor=(1.0, 1.0),
    )

    # --- Legenda waterlijnen ---
    line_handles = [
        plt.Line2D([0], [0], color=color, lw=2) for color in plotted_lines.values()
    ]
    line_labels = list(plotted_lines.keys())
    line_legend = ax.legend(
        line_handles,
        line_labels,
        title="Water lines",
        loc="upper right",
        bbox_to_anchor=(1.0, 0.6),
    )

    ax.add_artist(soil_legend)
    ax.add_artist(line_legend)

    ax.set_xlabel("X")
    ax.set_ylabel("Z")
    ax.set_title(f"Stage {stage_id} - {df_stage['stage_label'].iloc[0]}")
    ax.set_aspect("equal")
    plt.grid(True)
    ax.set_aspect(aspect=2.5)
    plt.show()

In [ ]:
def plot_stage(
    df_merged_soil: pd.DataFrame,
    df_merged_water: pd.DataFrame,
    df_merged_calculations: pd.DataFrame,
    stage_id,
    xlim,
    ylim,
):
    """
    Plot de geometrie van een stage als ingekleurde polygons per layer,
    kleuren gebaseerd op soil type, met waterlijnen en glijvlak circles.
    """
    stage_id = str(stage_id)
    df_stage = df_merged_soil[df_merged_soil["stage_id"] == stage_id]
    df_stage_water = df_merged_water[df_merged_water["stage_id"] == stage_id]
    df_stage_calc = df_merged_calculations[
        df_merged_calculations["stage_id"] == stage_id
    ]

    if df_stage.empty:
        print(f"Geen soil data gevonden voor stage {stage_id}")
        return
    if df_stage_water.empty:
        print(f"Geen water data gevonden voor stage {stage_id}")

    fig, ax = plt.subplots(figsize=(15, 15))

    # --- Plot soils ---
    plotted_soils = {}
    df_stage = df_stage.sort_values("soil_id")
    for idx, row in df_stage.iterrows():
        points = row["points"]
        if not points:
            continue
        polygon_coords = [(p["X"], p["Z"]) for p in points]
        if polygon_coords[0] != polygon_coords[-1]:
            polygon_coords.append(polygon_coords[0])
        color = row["color"]
        poly = Polygon(
            polygon_coords, closed=True, facecolor=color, edgecolor="k", alpha=1
        )
        ax.add_patch(poly)
        soil_name = row["name"]
        if soil_name not in plotted_soils:
            plotted_soils[soil_name] = color

    # --- Plot waterlijnen ---
    plotted_lines = {}
    for line_id, df_line in df_stage_water.groupby("line_id"):
        xs = df_line["x"].tolist()
        zs = df_line["z"].tolist()
        line_label = df_line["line_label"].iloc[0]
        color = df_line["color"].iloc[0]
        ax.plot(xs, zs, color=color, linewidth=2)
        if line_label not in plotted_lines:
            plotted_lines[line_label] = color

    # --- Plot glijvlakken (circles) ---
    plotted_circles = {}
    for idx, row in df_stage_calc.iterrows():
        if pd.notna(row.get("circle_center_x")) and pd.notna(row.get("circle_radius")):
            circle = Circle(
                (row["circle_center_x"], row["circle_center_z"]),
                row["circle_radius"],
                edgecolor="red",
                facecolor="none",
                linewidth=2,
                linestyle="--",
            )

        ax.add_patch(circle)
        analysis_type = row["analysis_type"]
        radius = row["circle_radius"]
        # Bewaar radius per analysis_type voor legenda
        if analysis_type not in plotted_circles:
            plotted_circles[analysis_type] = radius
        # clip_path toevoegen als gewenst
        ax.add_patch(circle)

        # Voeg toe aan dict voor legenda
        analysis_type = row["analysis_type"]
        if analysis_type not in plotted_circles:
            plotted_circles[analysis_type] = "red"

    # --- Legenda soils ---
    soil_handles = [
        plt.Line2D([0], [0], color=color, lw=10) for color in plotted_soils.values()
    ]
    soil_labels = list(plotted_soils.keys())
    soil_legend = ax.legend(
        soil_handles,
        soil_labels,
        title="Soil type",
        loc="upper right",
        bbox_to_anchor=(1.0, 1.0),
    )

    # --- Legenda waterlijnen ---
    line_handles = [
        plt.Line2D([0], [0], color=color, lw=2) for color in plotted_lines.values()
    ]
    line_labels = list(plotted_lines.keys())
    line_legend = ax.legend(
        line_handles,
        line_labels,
        title="Water lines",
        loc="upper right",
        bbox_to_anchor=(1.0, 0.6),
    )

    # --- Legenda glijvlakken ---
    circle_handles = [
        plt.Line2D([0], [0], color="red", lw=2, linestyle="--")
        for _ in plotted_circles.values()
    ]
    circle_labels = [
        f"Slip Circle ({atype}) – R={radius:.2f}"
        for atype, radius in plotted_circles.items()
    ]
    circle_legend = ax.legend(
        circle_handles,
        circle_labels,
        title="Slip Circles",
        loc="upper right",
        bbox_to_anchor=(1.0, 0.3),
    )

    ax.add_artist(soil_legend)
    ax.add_artist(line_legend)
    ax.add_artist(circle_legend)

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    ax.set_xlabel("X")
    ax.set_ylabel("Z")
    ax.set_title(f"Stage {stage_id}")
    ax.set_aspect("equal")
    plt.grid(True)
    plt.tight_layout()
    # ax.set_aspect(aspect=2.5)
    plt.show()

In [ ]:
plot_stage(
    df_merged_soil, df_merged_water, df_merged_calculations, "22", (20, 60), (-5, 5)
)